# L7 · Obtenir une sortie structurée 🧱

⏱️ **Durée : 35 à 40 minutes** · Niveau : débutant · Mistral + LangChain

Un modèle répond naturellement avec du texte. Une application, elle, a souvent besoin de champs prévisibles : un nom, un e-mail, un téléphone. Dans ce notebook, nous allons transformer une conversation libre en données directement exploitables par Python.

### 🎯 À la fin, vous saurez

- distinguer texte libre et sortie structurée ;
- décrire un contrat de données avec `TypedDict` puis Pydantic ;
- utiliser explicitement `ToolStrategy` avec un endpoint Mistral personnalisé ;
- inspecter `structured_response`, son type et les erreurs de validation.

> **Lien avec L6 :** la mémoire conserve les messages entre deux appels. L7 contrôle maintenant la **forme** des données extraites de ces messages.

## 🧠 Pourquoi structurer une réponse ?

Imaginez un appel téléphonique pris par un collègue :

- une **note libre** est agréable à lire, mais chaque information peut changer de place ;
- un **formulaire** possède toujours les mêmes cases et peut alimenter automatiquement un CRM.

Un LLM préfère converser. Notre programme préfère un contrat stable. La sortie structurée fait le pont entre les deux.

⚠️ Une structure ne garantit pas que l'information est vraie. Elle garantit seulement que la réponse respecte une **forme contrôlée**.

## 📖 Mini-glossaire

| Terme | Définition simple |
|---|---|
| **Schéma** | Description des champs attendus, de leurs types et parfois de leurs contraintes. |
| **`TypedDict`** | Contrat léger pour décrire les clés et types d'un dictionnaire. |
| **Pydantic** | Bibliothèque qui crée des objets Python et valide réellement leurs données à l'exécution. |
| **`ToolStrategy`** | Stratégie LangChain qui demande la structure via un appel d'outil compatible avec de nombreux modèles. |
| **`structured_response`** | Clé du résultat de l'agent où LangChain place l'objet structuré final. |

## 🗺️ Le trajet des données

```text
Conversation libre
      ↓
Schéma demandé par l'application
      ↓
Mistral extrait les valeurs
      ↓
ToolStrategy transporte la réponse structurée
      ↓
TypedDict ou Pydantic contrôle la forme
      ↓
result["structured_response"]
```

Dans ce parcours, **Mistral propose les valeurs**, **LangChain orchestre la stratégie**, puis **Python valide et manipule l'objet**.

## 🛠️ Préparer le modèle

Nous utilisons l'intégration officielle [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai) avec `mistral-medium-latest` et `temperature=0`.

Le notebook lit uniquement `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL` depuis le processus courant. Il ne charge, n'affiche et ne modifie aucun fichier `.env`.

In [1]:
import os

from langchain_mistralai import ChatMistralAI

MODEL = "mistral-medium-latest"


def normaliser_endpoint(url: str) -> str:
    """Retourne une URL de base terminée par /v1, sans révéler sa valeur."""
    base = url.strip().rstrip("/")
    return base if base.endswith("/v1") else f"{base}/v1"


variables_requises = ("MISTRAL_API_KEY", "MISTRAL_SERVER_URL")
manquantes = [nom for nom in variables_requises if not os.environ.get(nom)]
if manquantes:
    raise RuntimeError(
        "Variables d'environnement manquantes : " + ", ".join(manquantes)
    )

# 📚 Doc officielle LangChain :
# https://docs.langchain.com/oss/python/integrations/chat/mistralai
mistral_model = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.environ["MISTRAL_API_KEY"],
    endpoint=normaliser_endpoint(os.environ["MISTRAL_SERVER_URL"]),
)

print(f"✅ Modèle configuré : {MODEL} · temperature=0")

✅ Modèle configuré : mistral-medium-latest · temperature=0


> 👀 **Résultat attendu**
>
> Seuls le nom du modèle et sa température sont affichés. Si une variable manque, la cellule s'arrête sans révéler la clé ou l'URL.

## 🛠️ Notre cas : transformer une conversation en fiche contact

Le texte suivant reste volontairement oral et désordonné. Les informations utiles sont mêlées à une demande commerciale. Nous réutiliserons **exactement la même entrée** dans les trois approches afin que la comparaison soit honnête.

In [2]:
conversation_enregistree = """
Nous avons parlé avec John Doe. Il travaille chez Example. Son numéro est, voyons,
cinq cinq cinq, un deux trois, quatre cinq six sept. Vous l'avez bien noté ?
Son e-mail est john arobase example point com. Il souhaite commander 50 boîtes de céréales.
""".strip()

print(conversation_enregistree)

Nous avons parlé avec John Doe. Il travaille chez Example. Son numéro est, voyons,
cinq cinq cinq, un deux trois, quatre cinq six sept. Vous l'avez bien noté ?
Son e-mail est john arobase example point com. Il souhaite commander 50 boîtes de céréales.


## 1 · Baseline : demander du texte libre

La fonction [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) peut fonctionner sans `response_format`. Le modèle choisit alors librement les phrases, l'ordre et la ponctuation.

> 🧠 **Pause prédiction**
>
> Si nous exécutons deux fois l'extraction ou changeons de modèle, les trois informations seront-elles forcément placées aux mêmes endroits ?

In [3]:
from langchain.agents import create_agent

# 📚 Doc officielle LangChain :
# https://docs.langchain.com/oss/python/langchain/agents
agent_texte = create_agent(
    model=mistral_model,
    tools=[],
    system_prompt=(
        "Extrais le nom, l'e-mail et le téléphone de la conversation. "
        "Réponds en français."
    ),
)

resultat_texte = agent_texte.invoke(
    {"messages": [{"role": "user", "content": conversation_enregistree}]}
)
reponse_texte = resultat_texte["messages"][-1].content
print(reponse_texte)
print("Type Python :", type(reponse_texte))

Voici les informations extraites de la conversation :

- **Nom** : John Doe
- **E-mail** : john@example.com
- **Téléphone** : 05 55 12 34 56 77 (ou +33 5 55 12 34 56 77 selon le format souhaité)
Type Python : <class 'str'>


> 👀 **Résultat attendu**
>
> Le contenu mentionne John Doe, `john@example.com` et le numéro, mais son type Python reste `str`. La formulation exacte peut varier.

> 🔍 **Conséquence pour une application**
>
> Pour récupérer uniquement l'e-mail, il faudrait analyser le texte avec des règles fragiles. Une phrase différente pourrait casser ces règles. Il nous faut des cases nommées.

## 2 · Choisir explicitement `ToolStrategy`

LangChain documente deux grandes stratégies de [sortie structurée](https://docs.langchain.com/oss/python/langchain/structured-output) :

- la stratégie native du fournisseur quand le modèle et l'API l'annoncent ;
- `ToolStrategy`, qui représente le schéma comme un tool et s'appuie sur le tool calling.

Nous utilisons ici **explicitement `ToolStrategy`**. Pourquoi ? Un proxy Mistral personnalisé peut prendre en charge le tool calling sans annoncer toutes les métadonnées attendues pour une stratégie native. Ce choix rend la démonstration plus prévisible et évite une détection automatique dépendante de l'endpoint.

⚠️ `ToolStrategy` ne change pas notre fiche métier : elle change seulement la manière dont LangChain demande au modèle de la remplir.

### 🧱 Première fiche : `TypedDict`

`TypedDict` est léger : l'objet final reste un dictionnaire Python, mais ses clés et types sont déclarés. Les descriptions `Annotated` aident le modèle à comprendre le contenu attendu.

> 🧠 **Pause prédiction**
>
> Après l'invocation, `structured_response` sera-t-il une chaîne, un dictionnaire ou une instance de classe Pydantic ?

In [4]:
from typing import Annotated

from langchain.agents.structured_output import ToolStrategy
from typing_extensions import TypedDict


class ContactInfoDict(TypedDict):
    name: Annotated[str, "Nom complet de la personne"]
    email: Annotated[str, "Adresse e-mail normalisée"]
    phone: Annotated[str, "Numéro de téléphone composé uniquement de chiffres"]


# 📚 Doc officielle LangChain :
# https://docs.langchain.com/oss/python/langchain/structured-output
agent_typed_dict = create_agent(
    model=mistral_model,
    tools=[],
    response_format=ToolStrategy(ContactInfoDict),
    system_prompt=(
        "Extrais fidèlement la fiche contact. Normalise l'e-mail et écris le téléphone "
        "avec des chiffres uniquement."
    ),
)

In [5]:
from pprint import pprint

resultat_typed_dict = agent_typed_dict.invoke(
    {"messages": [{"role": "user", "content": conversation_enregistree}]}
)
contact_dict = resultat_typed_dict["structured_response"]

print("Type de structured_response :", type(contact_dict))
pprint(contact_dict)
print("E-mail accessible par sa clé :", contact_dict["email"])

Type de structured_response : <class 'dict'>
{'email': 'john@example.com', 'name': 'John Doe', 'phone': '5551234567'}
E-mail accessible par sa clé : john@example.com


> 👀 **Résultat attendu**
>
> `structured_response` est un `dict` avec trois clés stables :
>
> ```python
> {
>     "name": "John Doe",
>     "email": "john@example.com",
>     "phone": "5551234567",
> }
> ```
>
> 🔍 La valeur est immédiatement accessible avec `contact_dict["email"]`. Les annotations améliorent le contrat, mais `TypedDict` ne transforme pas ce dictionnaire en objet validé à l'exécution.

## 3 · Ajouter une validation réelle avec Pydantic

Un modèle Pydantic est comparable à un formulaire muni de contrôles : champ obligatoire, longueur minimale, format attendu. LangChain accepte directement un [`BaseModel` comme schéma de sortie](https://docs.langchain.com/oss/python/langchain/structured-output).

Nous utilisons une nouvelle classe `ContactInfoModel` au lieu d'écraser `ContactInfoDict`. La comparaison reste ainsi lisible.

In [6]:
from pydantic import BaseModel, Field, ValidationError


class ContactInfoModel(BaseModel):
    name: str = Field(min_length=1, description="Nom complet de la personne")
    email: str = Field(
        pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
        description="Adresse e-mail normalisée",
    )
    phone: str = Field(
        pattern=r"^\d{10}$",
        description="Numéro de téléphone sur exactement dix chiffres",
    )


print(ContactInfoModel.model_json_schema())

{'properties': {'name': {'description': 'Nom complet de la personne', 'minLength': 1, 'title': 'Name', 'type': 'string'}, 'email': {'description': 'Adresse e-mail normalisée', 'pattern': '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$', 'title': 'Email', 'type': 'string'}, 'phone': {'description': 'Numéro de téléphone sur exactement dix chiffres', 'pattern': '^\\d{10}$', 'title': 'Phone', 'type': 'string'}}, 'required': ['name', 'email', 'phone'], 'title': 'ContactInfoModel', 'type': 'object'}


### 🔍 Voir une erreur utile avant d'appeler le modèle

La validation appartient à Python : nous pouvons donc la tester localement, sans appel réseau.

> 🧠 **Pause prédiction**
>
> Combien de champs invalides Pydantic signalera-t-il pour un nom vide, un faux e-mail et un téléphone à deux chiffres ?

In [7]:
try:
    ContactInfoModel(name="", email="pas-un-email", phone="12")
except ValidationError as erreur:
    print("❌ Données refusées par Pydantic :")
    for detail in erreur.errors(include_url=False):
        champ = ".".join(str(partie) for partie in detail["loc"])
        print(f"- {champ}: {detail['msg']}")

❌ Données refusées par Pydantic :
- name: String should have at least 1 character
- email: String should match pattern '^[^@\s]+@[^@\s]+\.[^@\s]+$'
- phone: String should match pattern '^\d{10}$'


> 👀 **Résultat attendu :** trois erreurs, une par champ.

> 🔍 **Pourquoi est-ce utile ?**
>
> L'erreur indique précisément la case incorrecte et la contrainte violée. Avec `ToolStrategy`, LangChain peut également renvoyer une erreur de validation au modèle pour lui permettre de corriger sa proposition. Notre code doit néanmoins prévoir le cas où aucune réponse valide n'est obtenue.

### ▶️ Extraire un objet Pydantic

> 🧠 **Pause prédiction**
>
> Cette fois, `structured_response` doit-il être un dictionnaire ordinaire ou une instance de `ContactInfoModel` ? Quelle méthode permettra de le reconvertir en dictionnaire ?

In [8]:
agent_pydantic = create_agent(
    model=mistral_model,
    tools=[],
    response_format=ToolStrategy(ContactInfoModel),
    system_prompt=(
        "Extrais fidèlement la fiche contact. Normalise l'e-mail et écris le téléphone "
        "avec exactement dix chiffres."
    ),
)

resultat_pydantic = agent_pydantic.invoke(
    {"messages": [{"role": "user", "content": conversation_enregistree}]}
)
contact_model = resultat_pydantic["structured_response"]

print("Type de structured_response :", type(contact_model))
print("Objet :", contact_model)
print("Dictionnaire :", contact_model.model_dump())

Type de structured_response : <class '__main__.ContactInfoModel'>
Objet : name='John Doe' email='john@example.com' phone='0555123456'
Dictionnaire : {'name': 'John Doe', 'email': 'john@example.com', 'phone': '0555123456'}


> 👀 **Résultat attendu**
>
> Le type est `ContactInfoModel`. `model_dump()` produit un dictionnaire contenant John Doe, `john@example.com` et `5551234567`.

> 🔍 **Comparaison finale**
>
> | Approche | Objet Python | Forme stable | Validation à l'exécution | Usage typique |
> |---|---|---:|---:|---|
> | Texte libre | `str` | ❌ | ❌ | réponse destinée uniquement à un humain |
> | `TypedDict` | `dict` | ✅ | limitée | pipeline simple et léger |
> | Pydantic | `ContactInfoModel` | ✅ | ✅ | API, stockage, règles métier |

⚠️ Même validée, une adresse extraite peut être fausse. Pour des données sensibles, ajoutez des contrôles métier et une confirmation humaine.

## 🧪 Micro-exercice · Extraire aussi la commande

Créez `ContactOrderModel`, qui reprend la fiche Pydantic et ajoute :

- `company: str` pour l'entreprise ;
- `quantity: int` pour le nombre de boîtes, strictement positif.

Réutilisez la même conversation et `ToolStrategy`.

### ✅ Critères de réussite

- `structured_response` est une instance de `ContactOrderModel` ;
- `company` vaut `Example` et `quantity` vaut l'entier `50` ;
- une quantité égale à `0` est refusée par Pydantic.

💡 **Indice :** utilisez `Field(gt=0)` pour imposer une valeur positive.

In [9]:
# 👉 À vous : décommentez et remplacez les TODO.
# Le squelette reste commenté pour que « Run All » fonctionne avant l'exercice.
#
# class ContactOrderModel(BaseModel):
#     name: str = Field(min_length=1)
#     email: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
#     phone: str = Field(pattern=r"^\d{10}$")
#     company: str = TODO
#     quantity: int = TODO
#
# agent_commande = create_agent(
#     model=mistral_model,
#     response_format=ToolStrategy(TODO),
# )
# resultat_commande = agent_commande.invoke(
#     {"messages": [{"role": "user", "content": conversation_enregistree}]}
# )
# print(resultat_commande["structured_response"])

<details>
<summary>✅ Voir une correction complète</summary>

```python
class ContactOrderModel(BaseModel):
    name: str = Field(min_length=1)
    email: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    phone: str = Field(pattern=r"^\d{10}$")
    company: str = Field(min_length=1, description="Entreprise de la personne")
    quantity: int = Field(gt=0, description="Nombre de boîtes commandées")

agent_commande = create_agent(
    model=mistral_model,
    tools=[],
    response_format=ToolStrategy(ContactOrderModel),
    system_prompt=(
        "Extrais la fiche et la commande. Normalise l'e-mail et écris le téléphone "
        "avec exactement dix chiffres."
    ),
)
resultat_commande = agent_commande.invoke(
    {"messages": [{"role": "user", "content": conversation_enregistree}]}
)
commande = resultat_commande["structured_response"]
print(commande)
assert commande.company == "Example"
assert commande.quantity == 50

try:
    ContactOrderModel(
        name="John Doe",
        email="john@example.com",
        phone="5551234567",
        company="Example",
        quantity=0,
    )
except ValidationError:
    print("✅ Une quantité nulle est bien refusée.")
```

</details>

## 🧭 Ce qu'il faut retenir

Vous savez maintenant :

- ✅ reconnaître quand une chaîne libre est insuffisante pour une application ;
- ✅ récupérer le résultat structuré dans `result["structured_response"]` ;
- ✅ choisir `TypedDict` pour un dictionnaire léger ;
- ✅ choisir Pydantic pour valider des contraintes à l'exécution ;
- ✅ imposer `ToolStrategy` lorsque la détection native d'un proxy n'est pas fiable.

### ⚠️ Du notebook à la production

| Dans cette démonstration | En production |
|---|---|
| trois champs simples | schémas versionnés et compatibles avec les consommateurs |
| correction automatique possible | nombre de tentatives limité et erreur applicative claire |
| données fictives | validation métier, consentement et protection des données personnelles |
| une extraction | tests sur accents, champs absents et entrées malveillantes |

### 🧭 Transition vers L8

L7 impose une **forme de sortie**. Dans **L8**, nous adapterons les **instructions d'entrée** au contexte d'exécution : un même agent recevra un prompt différent selon le profil de l'utilisateur.

## 📚 Documentation officielle

- [LangChain · Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) — comparer stratégies fournisseur et `ToolStrategy`, schémas acceptés et gestion des erreurs.
- [LangChain · Agents](https://docs.langchain.com/oss/python/langchain/agents) — comprendre `create_agent` et l'état retourné.
- [LangChain · Intégration ChatMistralAI](https://docs.langchain.com/oss/python/integrations/chat/mistralai) — configurer Mistral dans LangChain.
- [Pydantic · Models](https://docs.pydantic.dev/latest/concepts/models/) — approfondir `BaseModel`, `Field`, `model_dump()` et les erreurs de validation.
- [Mistral · Function calling](https://docs.mistral.ai/studio/conversations/function-calling) — comprendre le mécanisme utilisé par `ToolStrategy`.